In [3]:
## Diabetes 
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

data = pd.read_csv("D:/smart decision/Data Preprocessing/Cleaned Datasets/cleaned_diabetic_data.csv")

X = data.drop("readmitted", axis=1)
y = data["readmitted"]

le = LabelEncoder()

for col in X.select_dtypes(include="object").columns:
    X[col] = le.fit_transform(X[col].astype(str))

y = le.fit_transform(y)


selector = SelectKBest(score_func=chi2, k=5)

X = selector.fit_transform(X, y)


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

sc = StandardScaler()

X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

final_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=2,
    random_state=42
)

final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))



Accuracy: 0.5847810706705447


In [7]:
## Saved the model
import joblib

joblib.dump(
    final_model,
    r"D:\smart decision\Final models\diabetic_readmission_model.pkl"
)

loaded_model = joblib.load(
    r"D:\smart decision\Final models\diabetic_readmission_model.pkl"
)

print("Model Saved Successfully!")

Model Saved Successfully!


In [1]:
## Pneumonia Final Model - Corrected Version

import os
import pandas as pd
import joblib

from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)


data = pd.read_csv(
    r"D:\smart decision\Data Preprocessing\Cleaned Datasets\cleaned_clinical_pneumonia_dataset.csv"
)

print("Dataset Shape:", data.shape)
print("\nColumns:")
print(data.columns.tolist())


X = data.drop("true_label", axis=1).copy()
y = data["true_label"].copy()

# Save original feature names
original_feature_names = X.columns.tolist()


feature_encoders = {}

categorical_columns = X.select_dtypes(
    include=["object", "category"]
).columns

for col in categorical_columns:

    encoder = LabelEncoder()

    X[col] = encoder.fit_transform(
        X[col].astype(str)
    )

    feature_encoders[col] = encoder


print("\nEncoded Categorical Columns:")
print(list(feature_encoders.keys()))



target_encoder = LabelEncoder()

y = target_encoder.fit_transform(
    y.astype(str)
)

print("\nTarget Class Mapping:")

for number, class_name in enumerate(target_encoder.classes_):
    print(number, "=", class_name)


selector = SelectKBest(
    score_func=chi2,
    k=5
)

X_selected = selector.fit_transform(X, y)

# Get exact selected feature names
selected_feature_names = X.columns[
    selector.get_support()
].tolist()


print("\n======================================")
print("SELECTED FEATURES FOR DASHBOARD")
print("======================================")

for feature in selected_feature_names:
    print(feature)

X_train, X_test, y_train, y_test = train_test_split(
    X_selected,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)


final_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=2,
    random_state=42
)

final_model.fit(
    X_train_scaled,
    y_train
)


y_pred = final_model.predict(
    X_test_scaled
)

print("\n======================================")
print("MODEL PERFORMANCE")
print("======================================")

print(
    "\nAccuracy:",
    accuracy_score(y_test, y_pred)
)

print(
    "\nConfusion Matrix:"
)

print(
    confusion_matrix(y_test, y_pred)
)

print(
    "\nClassification Report:"
)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=target_encoder.classes_
    )
)

SAVE_FOLDER = (
    r"D:\smart decision\Final models\Pneumoia"
)

os.makedirs(
    SAVE_FOLDER,
    exist_ok=True
)


joblib.dump(
    final_model,
    os.path.join(
        SAVE_FOLDER,
        "Pneumonia_model.pkl"
    )
)

joblib.dump(
    selector,
    os.path.join(
        SAVE_FOLDER,
        "Pneumonia_selector.pkl"
    )
)


joblib.dump(
    scaler,
    os.path.join(
        SAVE_FOLDER,
        "Pneumonia_scaler.pkl"
    )
)


joblib.dump(
    target_encoder,
    os.path.join(
        SAVE_FOLDER,
        "Pneumonia_target_encoder.pkl"
    )
)

joblib.dump(
    feature_encoders,
    os.path.join(
        SAVE_FOLDER,
        "Pneumonia_feature_encoders.pkl"
    )
)


joblib.dump(
    original_feature_names,
    os.path.join(
        SAVE_FOLDER,
        "Pneumonia_original_features.pkl"
    )
)


joblib.dump(
    selected_feature_names,
    os.path.join(
        SAVE_FOLDER,
        "Pneumonia_selected_features.pkl"
    )
)

loaded_model = joblib.load(
    os.path.join(
        SAVE_FOLDER,
        "Pneumonia_model.pkl"
    )
)

test_prediction = loaded_model.predict(
    X_test_scaled[:1]
)

actual_result = target_encoder.inverse_transform(
    test_prediction
)[0]


print("\n======================================")
print("ALL FILES SAVED SUCCESSFULLY!")
print("======================================")

print("\nSaved Model Test Prediction:")
print(actual_result)

print("\nSelected Dashboard Features:")
print(selected_feature_names)

print("\nTarget Classes:")
print(target_encoder.classes_)

Dataset Shape: (1500, 9)

Columns:
['timestamp', 'fever', 'tachycardia', 'crackles', 'oxygen_saturation', 'wbc_count', 'chest_xray_result', 'true_label', 'uncertainty_score']

Encoded Categorical Columns:
['timestamp', 'chest_xray_result']

Target Class Mapping:
0 = atelectasis
1 = pneumonia
2 = pulmonary edema

SELECTED FEATURES FOR DASHBOARD
timestamp
tachycardia
crackles
chest_xray_result
uncertainty_score

MODEL PERFORMANCE

Accuracy: 0.696

Confusion Matrix:
[[ 87  15  23]
 [  7 113   5]
 [ 38  26  61]]

Classification Report:
                 precision    recall  f1-score   support

    atelectasis       0.66      0.70      0.68       125
      pneumonia       0.73      0.90      0.81       125
pulmonary edema       0.69      0.49      0.57       125

       accuracy                           0.70       375
      macro avg       0.69      0.70      0.69       375
   weighted avg       0.69      0.70      0.69       375


ALL FILES SAVED SUCCESSFULLY!

Saved Model Test Prediction:

In [1]:
## Length of Stay Final Model

import pandas as pd
import joblib
import numpy as np

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

data = pd.read_csv(r"D:\smart decision\Data Preprocessing\Cleaned Datasets\cleaned_LengthOfStay.csv")

X = data.drop("lengthofstay", axis=1)
y = data["lengthofstay"]

le = LabelEncoder()

for col in X.select_dtypes(include="object").columns:
    X[col] = le.fit_transform(X[col].astype(str))


X_train, X_test, y_train, y_test = train_test_split( X, y,test_size=0.25,random_state=42)


sc = StandardScaler()

X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)


final_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=2,
    random_state=42
)

final_model.fit(X_train, y_train)


y_pred = final_model.predict(X_test)

print("MAE :", mean_absolute_error(y_test, y_pred))
print("MSE :", mean_squared_error(y_test, y_pred))
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score :", r2_score(y_test, y_pred))


joblib.dump(
    final_model,
    r"D:\smart decision\Final models\Length of stay\lengthofstay_model.pkl"
)

loaded_model = joblib.load(
    r"D:\smart decision\Final models\Length of stay\lengthofstay_model.pkl"
)

print("Length of Stay Model Saved Successfully!")

print("Saved Successfully!")

MAE : 0.6381967606631963
MSE : 0.7211511070720268
RMSE : 0.8492061628792074
R2 Score : 0.8687058823453968
Length of Stay Model Saved Successfully!
Saved Successfully!


In [7]:
## Hospital Analysis Final Model

import pandas as pd
import joblib

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

data = pd.read_csv(r"D:\smart decision\Data Preprocessing\Cleaned Datasets\cleaned_hospital_data_analysis.csv")


X = data.drop("Outcome", axis=1)
y = data["Outcome"]


le = LabelEncoder()

for col in X.select_dtypes(include="object").columns:
    X[col] = le.fit_transform(X[col].astype(str))

y = le.fit_transform(y)


selector = SelectKBest(score_func=chi2, k=5)

X = selector.fit_transform(X, y)


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)


sc = StandardScaler()

X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)


final_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=2,
    random_state=42
)

final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)


print("Accuracy :", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

joblib.dump(
    final_model,
    r"D:\smart decision\Final models\hospital_model.pkl"
)

loaded_model = joblib.load(
    r"D:\smart decision\Final models\hospital_model.pkl"
)

print("Hospital Analysis Model Saved Successfully!")


joblib.dump(
    final_model,
    r"D:\smart decision\Final models\hospital_model.pkl"
)

loaded_model = joblib.load(
    r"D:\smart decision\Final models\hospital_model.pkl"
)

print("Hospital Analysis Model Saved Successfully!")

Accuracy : 1.0
[[144   0]
 [  0 102]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       144
           1       1.00      1.00      1.00       102

    accuracy                           1.00       246
   macro avg       1.00      1.00      1.00       246
weighted avg       1.00      1.00      1.00       246

Hospital Analysis Model Saved Successfully!
Hospital Analysis Model Saved Successfully!
